# Agent:交互范式的质变

Agent是一种具备环境感知、自主规划与任务执行能力的智能系统，是目标驱动的，不再死守预设的步骤

一个完整的Agent的支撑体系

- LLM：相当于Agent的大脑，通过其强大的语义理解、推理与泛化能力，让Agent在动态环境下自主规划路径、调用工具并实施修正策略
- 感知模块：相当于Agent的感官，通过API、实时检索，与文件系统接入多维外部信息
- 工具集：相当于Agent的四肢，通过Function Calling与MCP，将逻辑影响力延伸至真实软件环境
- 记忆模块：相当于Agent的经验库，利用向量数据库或记忆流维护上下文连贯性，支撑长周期的决策规划

# Function Calling:AI连接现实世界的接口

LLM可以根据用户意图，结构化地选择工具并生成调用参数

eg：赋予AI实时天气查询能力

主要包含以下四个环节：

- 意图识别
- 工具检索
- 参数映射
- 执行闭环

In [ ]:
import os
import json
import httpx
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    http_client=httpx.Client(verify=False)
)

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取制定城市的天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称,支持中文或英文"
                    }
                },
                "required": ["city"]
            }
        }
    }
]


def get_weather(city):
    url = f"https://wttr.in/{city}?format=3&lang=zh-cn"

    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.text.strip()
    except Exception as e:
        return f"获取天气信息失败:{e}"


def run(messages):
    response = client.chat.completions.create(
        model="qwen3.8-flash",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    if message.tool_calls:
        tool_call = message.tool_calls[0]
        fn_name = tool_call.function.name
        fn_args = json.loads(tool_call.function.arguments)

        print(f"模型调用函数:{fn_name}({fn_args})")

        if fn_name == "get_weather":
            result = get_weather(**fn_args)
        else:
            result = f"未知函数:{fn_name}"

        messages.append(message)
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            }
        )

        final_response = client.chat.completions.create(
            model="qwen3.8-flash",
            messages=messages
        )

        return final_response.choices[0].message.content

    return message.content


messages = [
    {"role": "system", "content": "你是一个有用的助手，可以查询天气信息。"},
    {"role": "user", "content": "北京今天天气怎么样？"}
]

answer = run(messages)
answer

# Agent Loop:Agent的核心

现实中的任务很少像查询天气这样，依次调用就能完成。因此，我们需要引入Agent Loop,通过循环机制来处理复杂的用户任务

Agent Loop可以将复杂目标拆解为可追踪的执行步骤：每次完成动作后，会实时捕获工具返回的结果，让模型进行反思，并据此动态调整后续路径

AgentLoop是Agent(如Codex，OpenClaw等)必须具备的核心运作机制

eg:手搓计算器Agent

In [ ]:
import re

# 定义计算工具函数
def multiply(a, b):
    """乘法工具"""
    return float(a) * float(b)


def divide(a, b):
    """除法工具"""
    if float(b) == 0:
        raise ValueError("除数不能为零")
    return float(a) / float(b)


# 解析工具调用
def parse_tool_call(tool_call):
    """解析工具调用指令"""
    pattern = r'<tool>([^<]*)</tool>'
    match = re.search(pattern, tool_call)
    if match:
        return match.group(1).strip()
    return None


# 执行工具调用
def execute_tool(tool_command):
    """执行工具命令并返回结果"""
    try:
        # 解析函数名和参数
        if tool_command.startswith("multiply("):
            # 提取参数
            args_match = re.match(r'multiply\(([^,]+),\s*([^)]+)\)', tool_command)
            if args_match:
                a, b = args_match.groups()
                result = multiply(a, b)
                return f"<tool_output>{result}</tool_output>"

        elif tool_command.startswith("divide("):
            # 提取参数
            args_match = re.match(r'divide\(([^,]+),\s*([^)]+)\)', tool_command)
            if args_match:
                a, b = args_match.groups()
                result = divide(a, b)
                return f"<tool_output>{result}</tool_output>"

        return f"<tool_output>未知工具调用: {tool_command}</tool_output>"

    except Exception as e:
        return f"<tool_output>工具执行错误: {str(e)}</tool_output>"


# 打印发送给AI的请求内容
def print_messages(messages, iteration):
    print("\n" + "=" * 50)
    print(f"第 {iteration} 次API调用, 请求内容：")
    print("=" * 50)

    # 打印整个messages的JSON内容
    print(json.dumps(messages, indent=4, ensure_ascii=False))

    print("=" * 50)

load_dotenv()

# 调用大模型
def invoke_model(user_input):
    # 初始化客户端
    client = OpenAI(
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        http_client=httpx.Client(verify=False)
    )

    # 工具使用说明
    tool_use = """
    当用户问计算相关问题，有必要可以使用工具，每个工具都是函数。
    使用工具的方式为输出 "<tool>使用工具指令</tool>"。
    使用工具后，你要等待回传结果 "<tool_output>工具回传的结果</tool_output>"。
    最后，输出最终结果"<final_result>最终答案</final_result>"

    可用工具：
    multiply(a,b): 返回 a 乘以 b
    divide(a,b): 返回 a 除以 b
    """

    # 初始化
    messages = [
        {"role": "system", "content": tool_use},
        {"role": "user", "content": user_input},
    ]

    max_iterations = 5  # 防止无限循环
    iteration = 0
    while iteration < max_iterations:
        iteration += 1

        print_messages(messages, iteration)

        # 调用API
        response = client.chat.completions.create(
            model="qwen3.6-flash",
            messages=messages,
            stream=False
        )

        assistant_content = response.choices[0].message.content
        print(f"\nAI回复: {assistant_content}")

        # 检查是否需要工具调用
        tool_command = parse_tool_call(assistant_content)

        if tool_command:
            print(f"执行工具: {tool_command}")
            tool_result = execute_tool(tool_command)
            print(f"工具结果: {tool_result}")

            # 将工具结果添加到对话历史
            messages.append({"role": "assistant", "content": assistant_content})
            messages.append({"role": "user", "content": tool_result})

        else:
            # 检查是否有最终结果
            final_match = re.search(r'<final_result>([^<]*)</final_result>', assistant_content)
            if final_match:
                final_answer = final_match.group(1).strip()
                print(f"\n🎉 最终答案: {final_answer}")
            break
    if iteration >= max_iterations:
        print("⚠️  计算过程过于复杂，请简化您的问题。")


# 主函数
def main():
    print("AI计算助手已启动！输入计算问题或输入'退出'结束程序。")

    while True:
        # 获取用户输入
        user_input = input("\n请输入计算问题: ").strip()

        if user_input.lower() in ['退出', 'exit', 'quit']:
            print("再见！")
            break

        if not user_input:
            continue

        invoke_model(user_input)


if __name__ == "__main__":
    main()

# ReAct:通用的任务处理框架

ReAct将Agent的思维过程从文字创作约束为标准化的逻辑循环，在该循环中，思考、行动、观察构成三位一体的逻辑链

1. 思考：深度思考。Agent暂停执行，开始思考两个问题
    - 进度对齐：根据已有执行记录，当前进展到那一步
    - 差距分析：为了达成目标，最欠缺什么信息
2. 行动：精准执行。这是Agent连接现实世界的唯一方式。基于思考结论，从工具箱中调用具体指令。行动必须满足两个要求
    - 确定性：必须是具体的、可参数化的命令，严禁模糊意图
    - 最小化：倾向于执行”最小原子操作“，以便出错时快速回溯
3. 观察：环境反馈。这是Agent的数字触觉，也是消除幻觉的关键。观察具有两重价值
    - 客观真实：反馈来自编译器报错、测试失败日志或文件系统的真实状态，而非模型推测
    - 输入转化：模型接收原始反馈后，将其转化为下一轮思考的关键变量

三者形成闭环：思考决定行动，行动产生观察，观察驱动下一轮思考。这个循环持续进行，直至任务完成或人工中断

eg. 从零开始构建一个简易版编程助手

## 编写宪法：ReAct System Prompt

Agent的所有理智与行为边界，都源于其System Prompt

为了让LLM摆脱纯文本生成的惯性，我们需要定义一套严格的XML协议，强制模型遵循”思考-行动-观察“的循环来输出

In [ ]:
react_system_prompt_template = """
你需要解决一个问题。为此，你需要将问题分解为多个步骤。对于每个步骤，首先使用 <thought> 思考要做什么，然后使用可用工具之一决定一个 <action>。接着，你将根据你的行动从环境/工具中收到一个 <observation>。持续这个思考和行动的过程，直到你有足够的信息来提供 <final_answer>。
所有步骤请严格使用以下 XML 标签格式输出：
- <question> 用户问题
- <thought> 思考
- <action> 采取的工具操作
- <observation> 工具或环境返回的结果
- <final_answer> 最终答案
⸻
例子 1:
<question>埃菲尔铁塔有多高？</question>
<thought>我需要找到埃菲尔铁塔的高度。可以使用搜索工具。</thought>
<action>get_height("埃菲尔铁塔")</action>
<observation>埃菲尔铁塔的高度约为330米（包含天线）。</observation>
<thought>搜索结果显示了高度。我已经得到答案了。</thought>
<final_answer>埃菲尔铁塔的高度约为330米。</final_answer>
⸻
请严格遵守：
- 你每次回答都必须包括两个标签，第一个是 <thought>，第二个是 <action> 或 <final_answer>
- 输出 <action> 后立即停止生成，等待真实的 <observation>，擅自生成 <observation> 将导致错误
- 如果 <action> 中的某个工具参数有多行的话，请使用 \n 来表示，如：<action>write_to_file("/tmp/test.txt", "a\nb\nc")</action>
- 工具参数中的文件路径请使用绝对路径，不要只给出一个文件名。比如要写 write_to_file("/tmp/test.txt", "内容")，而不是 write_to_file("test.txt", "内容")
⸻
本次任务可用工具：
${tool_list}
⸻
环境信息：
操作系统：${operating_system}
工作目录：${working_directory}
工作目录下文件列表：${file_list}
"""

- 强制思考：要求模型在执行任何具体操作前，必须先进行逻辑推演。这不仅能有效减少幻觉，还能让开发者清晰地看到AI的决策路径，极大地提升了系统的透明度和可调试性
- 标准化行动：通过规定的too_name("arg")的格式，将自然语言转化为机器可解析的指令
- 环境感知：在提示词中动态注入当前的操作系统、工作目录及文件列表，让Agent能根据真实的物理环境做出准确判断

## 实现核心调度器：ReActAgent类

ReActAgent类是访问LLM并利用ReAct循环处理函数调用的类。其核心执行流程包含以下4个步骤

- 初始化：加载通义千问（DashScope OpenAI兼容接口）等底层大模型服务
- 渲染：将当前目录下的真实文件列表动态注入提示词，让Agent具备环境感知能力
- 循环：实时监测LLM的输出，一旦检测到<action>标签，就利用Python内置函数getattr或函数映射机制执行本地工具
- 解析：精准处理多行字符串转义，确保模型写入的代码格式完全正确

In [ ]:
import ast
import inspect
import platform
from string import Template
from typing import List, Callable, Tuple

class ReActAgent:
    def __init__(self, tools: List[Callable], model: str = "qwen-plus", project_directory: str = None):
        self.tools = {func.__name__: func for func in tools}
        self.model = model
        self.project_directory = os.path.abspath(project_directory or os.getcwd())
        self.client = OpenAI(
            api_key=os.getenv("DASHSCOPE_API_KEY"),
            base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
            http_client=httpx.Client(verify=False, trust_env=False),
        )


    # 打印发送给AI的请求内容
    def print_messages(self, messages, description: str):
        print("\n" + "=" * 50)
        print(description)
        print("=" * 50)
        # 打印整个messages的JSON内容
        print(json.dumps(messages, indent=4, ensure_ascii=False))
        print("=" * 50)

    def run(self, user_input: str):
        messages = [
            {"role": "system", "content": self.render_system_prompt(react_system_prompt_template)},
            {"role": "user", "content": f"<question>{user_input}</question>"}
        ]

        max_iterations = 10  # 防止无限循环
        iteration = 0

        while iteration < max_iterations:
            iteration += 1
            # 打印请求
            self.print_messages(messages, f"第 {iteration} 次API调用, 请求内容：")

            # 请求模型
            content = self.call_model(messages)

            # 打印返回
            self.print_messages(content, f"第 {iteration} 次API调用, 返回内容：")

            # 检测 Thought
            thought_match = re.search(r"<thought>(.*?)</thought>", content, re.DOTALL)
            if thought_match:
                thought = thought_match.group(1)
                print(f"\n\n💭 Thought: {thought}")

            # 检测模型是否输出 Final Answer，如果是的话，直接返回
            if "<final_answer>" in content:
                final_answer = re.search(r"<final_answer>(.*?)</final_answer>", content, re.DOTALL)
                return final_answer.group(1)

            # 检测 Action
            action_match = re.search(r"<action>(.*?)</action>", content, re.DOTALL)
            if not action_match:
                raise RuntimeError("模型未输出 <action>")
            action = action_match.group(1)
            tool_name, args = self.parse_action(action)

            print(f"\n\n🔧 Action: {tool_name}({', '.join(str(a) for a in args)})")
            # 只有终端命令才需要询问用户，其他的工具直接执行
            should_continue = input(f"\n\n是否继续？（Y/N）") if tool_name == "run_terminal_command" else "y"
            if should_continue.lower() != 'y':
                print("\n\n操作已取消。")
                return "操作被用户取消"

            try:
                observation = self.tools[tool_name](*args)
            except Exception as e:
                observation = f"工具执行错误：{str(e)}"
            print(f"\n\n🔍 Observation：{observation}")
            obs_msg = f"<observation>{observation}</observation>"
            messages.append({"role": "user", "content": obs_msg})

        return "⚠️ 已达到最大循环次数，任务未完成，请简化问题后重试。"

    def get_tool_list(self) -> str:
        """生成工具列表字符串，包含函数签名和简要说明"""
        tool_descriptions = []
        for func in self.tools.values():
            name = func.__name__
            signature = str(inspect.signature(func))
            doc = inspect.getdoc(func)
            tool_descriptions.append(f"- {name}{signature}: {doc}")
        return "\n".join(tool_descriptions)

    def render_system_prompt(self, system_prompt_template: str) -> str:
        """渲染系统提示模板，替换变量"""
        tool_list = self.get_tool_list()
        file_list = ", ".join(
            os.path.abspath(os.path.join(self.project_directory, f))
            for f in os.listdir(self.project_directory)
        )
        return Template(system_prompt_template).substitute(
            operating_system=self.get_operating_system_name(),
            working_directory=self.project_directory,
            tool_list=tool_list,
            file_list=file_list
        )

    def call_model(self, messages):
        print("\n\n正在请求模型，请稍等...")
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
        )
        content = response.choices[0].message.content
        messages.append({"role": "assistant", "content": content})
        return content

    def parse_action(self, code_str: str) -> Tuple[str, List[str]]:
        match = re.match(r'(\w+)\((.*)\)', code_str, re.DOTALL)
        if not match:
            raise ValueError("Invalid function call syntax")

        func_name = match.group(1)
        args_str = match.group(2).strip()

        # 手动解析参数，特别处理包含多行内容的字符串
        args = []
        current_arg = ""
        in_string = False
        string_char = None
        i = 0
        paren_depth = 0

        while i < len(args_str):
            char = args_str[i]

            if not in_string:
                if char in ['"', "'"]:
                    in_string = True
                    string_char = char
                    current_arg += char
                elif char == '(':
                    paren_depth += 1
                    current_arg += char
                elif char == ')':
                    paren_depth -= 1
                    current_arg += char
                elif char == ',' and paren_depth == 0:
                    # 遇到顶层逗号，结束当前参数
                    args.append(self._parse_single_arg(current_arg.strip()))
                    current_arg = ""
                else:
                    current_arg += char
            else:
                current_arg += char
                if char == string_char and (i == 0 or args_str[i - 1] != '\\'):
                    in_string = False
                    string_char = None

            i += 1

        # 添加最后一个参数
        if current_arg.strip():
            args.append(self._parse_single_arg(current_arg.strip()))

        return func_name, args

    def _parse_single_arg(self, arg_str: str):
        """解析单个参数"""
        arg_str = arg_str.strip()

        # 如果是字符串字面量
        if (arg_str.startswith('"') and arg_str.endswith('"')) or \
                (arg_str.startswith("'") and arg_str.endswith("'")):
            # 移除外层引号并处理转义字符
            inner_str = arg_str[1:-1]
            # 处理常见的转义字符
            inner_str = inner_str.replace('\\"', '"').replace("\\'", "'")
            inner_str = inner_str.replace('\\n', '\n').replace('\\t', '\t')
            inner_str = inner_str.replace('\\r', '\r').replace('\\\\', '\\')
            return inner_str

        # 尝试使用 ast.literal_eval 解析其他类型
        try:
            return ast.literal_eval(arg_str)
        except (SyntaxError, ValueError):
            # 如果解析失败，返回原始字符串
            return arg_str

    def get_operating_system_name(self):
        os_map = {
            "Darwin": "macOS",
            "Windows": "Windows",
            "Linux": "Linux"
        }

        return os_map.get(platform.system(), "Unknown")

In [ ]:
def read_file(file_path):
    """用于读取文件内容"""
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()


def write_to_file(file_path, content):
    """将指定内容写入指定文件"""
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content.replace("\\n", "\n"))
    return "写入成功"


def run_terminal_command(command):
    """用于执行终端命令"""
    import subprocess
    run_result = subprocess.run(command, shell=True, capture_output=True, text=True)
    return "执行成功" if run_result.returncode == 0 else run_result.stderr


tools = [read_file, write_to_file, run_terminal_command]

# 工作目录默认为notebook当前目录下的一个隔离子目录，避免Agent写文件污染项目
workspace = os.path.join(os.getcwd(), "react_workspace")
os.makedirs(workspace, exist_ok=True)

agent = ReActAgent(
    tools=tools,
    model="qwen-plus",  # 可替换为 qwen-max / qwen-turbo
    project_directory=workspace,
)

task = input("请输入任务：")

final_answer = agent.run(task)
print(f"\n\n✅ Final Answer：{final_answer}")